# 🤖 NeuroDiverAgents - Neurodivergent Parenting Support Agents
## Kaggle Agents Intensive - Capstone Project

**Author**: Marcus Carvalho  
**Track**: 🌍 Agents for Good  
**Course**: Google AI Agents Development Kit (ADK) - 5 Day Intensive  
**Project Focus**: Multi-agent system supporting parents of neurodivergent children (ADHD + ASD Level 1)

---

## 📋 Project Overview

This capstone demonstrates **6+ ADK capabilities** through a real-world application:

| Day | Capability | Implementation |
|-----|------------|----------------|
| **Day 1** | Multi-agent hierarchical system | 1 Manager + 5 Specialist Agents |
| **Day 2** | Tools & MCP | GoogleSearchTool, Custom FunctionTools, AgentTools |
| **Day 3** | Memory & Context | Cross-session learning, personalization |
| **Day 4** | Session Management | Conversation context, state tracking |
| **Day 5** | Production-Ready | Error handling, retry strategies, type safety |

### 🎯 Problem Statement

Parents of neurodivergent children face complex challenges:
- Is this behavior ADHD, autism, or typical 8-year-old development?
- What evidence-based strategies work for ADHD executive function challenges?
- How do I structure activities that support my child's specific needs?
- What patterns work for MY child based on past experiences?

### 💡 Solution

A **multi-agent AI system** that combines:
- **Specialist expertise** (ADHD, ASD, developmental psychology)
- **Evidence-based research** (Google Search integration)
- **Personalized learning** (pattern analysis from session history)
- **Practical tools** (behavior classification, activity planning)

---

## 🏗️ System Architecture

### Agent Hierarchy

```
┌─────────────────────────────────────┐
│  Parenting Coordinator (Manager)   │
│  - Orchestrates specialists         │
│  - Routes questions intelligently   │
│  - Synthesizes multi-perspective    │
└────────────┬────────────────────────┘
             │
     ┌───────┴───────────┬─────────────┬─────────────┬──────────────┐
     │                   │             │             │              │
┌────▼────┐      ┌──────▼─────┐  ┌───▼────┐   ┌───▼─────┐  ┌────▼──────┐
│  ADHD   │      │    ASD     │  │  Dev   │   │ Memory  │  │ Activity  │
│ Expert  │      │  Expert    │  │ Expert │   │  Agent  │  │  Planner  │
└─────────┘      └────────────┘  └────────┘   └─────────┘  └───────────┘
```

### Specialist Agents

1. **ADHD Expert** 🧠
   - Executive function challenges (working memory, task initiation)
   - Attention regulation and impulse control
   - Tools: GoogleSearchTool + BehaviorClassifier

2. **ASD Expert** 🎯
   - Sensory processing differences
   - Communication patterns and social interaction
   - Tools: GoogleSearchTool + BehaviorClassifier

3. **Developmental Expert** 📚
   - Age-appropriate expectations for 8-year-olds
   - Typical developmental milestones
   - Tools: GoogleSearchTool

4. **Memory Agent** 🧩
   - Pattern learning from session history
   - Personalized recommendations
   - Tools: PatternAnalyzer (custom)

5. **Activity Planner** 📋
   - Structured activity plans with materials
   - Executive function support
   - Tools: ActivityPlanner (custom)

### Type Safety Architecture

**All domain data uses Pydantic v2 models** with strict validation:
- ✅ No `Dict[str, Any]` for domain data
- ✅ Discriminated unions for results (Success | Error)
- ✅ Enums for constants (no string typos)
- ✅ Field validation with min/max constraints
- ✅ ConfigDict with strict mode enabled

---

## ⚙️ Setup & Installation

### Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q google-genai google-adk pydantic pydantic-settings

print("✅ Dependencies installed successfully!")

### Step 2: Configure Google API Key

**⚠️ Important**: You need a Google AI API key. Get one at: https://aistudio.google.com/app/apikey

For Kaggle notebooks, add your API key as a **Secret** named `GOOGLE_API_KEY` in the notebook settings.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

# Retrieve API key from Kaggle secrets
try:
    user_secrets = UserSecretsClient()
    os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GOOGLE_API_KEY")
    print("✅ API key configured from Kaggle secrets")
except Exception:
    print("⚠️ Kaggle secrets not available. Checking environment variable...")
    if not os.getenv("GOOGLE_API_KEY"):
        print("❌ Error: GOOGLE_API_KEY not found!")
        print("Please add it as a Kaggle secret or set it manually:")
        print('  os.environ["GOOGLE_API_KEY"] = "your-api-key-here"')
    else:
        print("✅ API key found in environment")

### Step 3: Upload Project Files

Upload the `src/` directory containing all agent and model code to your Kaggle notebook's working directory.

**Directory structure needed:**
```
/kaggle/working/
└── src/
    └── capstone/
        ├── models/
        ├── tools/
        └── agents/
```

In [ ]:
import sys
from pathlib import Path

# Add src directory to Python path
working_dir = Path.cwd()
src_dir = working_dir / "src"

if src_dir.exists():
    sys.path.insert(0, str(working_dir))
    print(f"✅ Added {working_dir} to Python path")
    print(f"✅ Source directory found: {src_dir}")
else:
    print(f"❌ Error: {src_dir} not found!")
    print("Please upload the 'src/' directory from the project.")

### Step 4: Import and Initialize System

In [ ]:
import asyncio
import re

from google import genai
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# Import our multi-agent system
from src.capstone.agents import create_coordinator

print("✅ All imports successful!")
print("✅ Ready to initialize multi-agent system")

---

## 🛠️ Helper Functions

In [ ]:
def print_separator(title: str) -> None:
    """Print formatted section separator."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80 + "\n")


def extract_retry_delay(error_message: str) -> float:
    """Extract retry delay in seconds from error message."""
    match = re.search(r"retry in (\d+\.?\d*)s", error_message)
    if match:
        return float(match.group(1))

    match = re.search(r"retryDelay.*?(\d+)s", error_message)
    if match:
        return float(match.group(1))

    return 10.0  # Default to 10 seconds


async def run_scenario_with_retry(
    runner: Runner,
    user_id: str,
    session_id: str,
    message: genai.types.Content,
    max_retries: int = 3,
) -> None:
    """Run a scenario with automatic retry on 429 errors."""
    for attempt in range(max_retries):
        try:
            print("💬 Response:")
            response_received = False

            async for event in runner.run_async(
                user_id=user_id,
                session_id=session_id,
                new_message=message,
            ):
                if hasattr(event, "response") and event.response:
                    print(event.response)
                    response_received = True

            if response_received:
                return  # Success!

        except Exception as e:
            error_str = str(e)

            if "429" in error_str and "RESOURCE_EXHAUSTED" in error_str:
                retry_delay = extract_retry_delay(error_str)

                if attempt < max_retries - 1:
                    print(
                        f"\n⏳ Rate limit hit. Waiting {retry_delay:.1f}s before retry (attempt {attempt + 1}/{max_retries})..."
                    )
                    await asyncio.sleep(retry_delay)
                    print("🔄 Retrying...\n")
                else:
                    print(f"\n❌ Rate limit exceeded after {max_retries} attempts.")
                    print("💡 Tips to resolve:")
                    print("   • Wait a few minutes before running again")
                    print("   • Check your API quota at: https://ai.dev/usage?tab=rate-limit")
                    return
            else:
                print(f"❌ Error: {error_str}")
                return


print("✅ Helper functions defined")

---

## 🚀 Initialize Multi-Agent System

Creates the **Parenting Coordinator** (manager) which automatically creates all 5 specialist agents.

In [ ]:
print_separator("🤖 Neurodivergent Parenting Support Agent - ADK Demo")
print("Demonstrating multi-agent hierarchical system with:")
print("  • Parenting Coordinator (Manager)")
print("  • 5 Specialist Agents (ADHD, ASD, Developmental, Memory, Activity Planner)")
print("  • Custom tools (Behavior Classifier, Activity Planner, Pattern Analyzer)")
print("  • Google Search integration")
print(
    "\n💡 Note: Retry handling follows ADK best practices with agent-level retries + application-level fallback."
)

# Create coordinator (which creates all specialists)
print("\n🔧 Initializing multi-agent system...")
coordinator = create_coordinator()
print("✅ All agents initialized!")

# Create runner with session service
session_service = InMemorySessionService()
runner = Runner(
    app_name="agents",
    agent=coordinator,
    session_service=session_service,
)
print("✅ Runner configured with session management")
print("\n🎬 Ready to run demo scenarios!")

---

## 📋 Demo Scenario 1: Homework Refusal Analysis

**Demonstrates:**
- Multi-agent orchestration (coordinator delegates to specialists)
- Behavior classification tool
- Google Search integration for evidence-based strategies
- Cross-specialist synthesis

**Parent Question:**
> "My 8-year-old son refuses to start his homework after school. He just played video games for an hour and now says he's 'too tired' for homework. When I try to get him started, he gets frustrated and sometimes has a meltdown. This happens most evenings around 5-6pm. Is this ADHD, autism, or just being 8?"

In [ ]:
print_separator("📋 SCENARIO 1: Homework Refusal Analysis")

scenario_1 = """My 8-year-old son refuses to start his homework after school.
He just played video games for an hour and now says he's "too tired" for homework.
When I try to get him started, he gets frustrated and sometimes has a meltdown.
This happens most evenings around 5-6pm. Is this ADHD, autism, or just being 8?"""

print("Parent Question:")
print(f'"{scenario_1}"\n')
print("🤖 Coordinator orchestrating specialists...\n")

# Create session for scenario 1
await session_service.create_session(
    app_name="agents",
    user_id="demo_user",
    session_id="scenario_1",
)

# Run scenario with automatic retry
await run_scenario_with_retry(
    runner=runner,
    user_id="demo_user",
    session_id="scenario_1",
    message=genai.types.Content(parts=[genai.types.Part(text=scenario_1)]),
)

print("\n✅ Scenario 1 complete!")

### 🔍 What Just Happened?

1. **Coordinator received** the parent's question
2. **Identified relevant specialists**: ADHD Expert, Developmental Expert, possibly ASD Expert
3. **Delegated** to specialists using AgentTool pattern
4. **Each specialist**:
   - Used BehaviorClassifier tool to analyze ADHD/ASD/age-typical factors
   - Searched Google for evidence-based strategies (CHADD, CDC resources)
   - Provided specialist perspective
5. **Coordinator synthesized** insights into comprehensive response

**ADK Capabilities Demonstrated:**
- ✅ Hierarchical agent orchestration
- ✅ Agent-as-tool (AgentTool)
- ✅ Custom function tools (BehaviorClassifier)
- ✅ Google Search integration
- ✅ Session management

In [ ]:
# Add delay between scenarios to avoid rate limits
print("\n⏳ Waiting 2 seconds before next scenario...\n")
await asyncio.sleep(2)

---

## 📋 Demo Scenario 2: Bedtime Routine Planning

**Demonstrates:**
- Activity planning with structured outputs
- Material specification and environmental setup
- Success criteria definition
- Integration with past experiences (if memory available)

**Parent Question:**
> "I need help creating a structured bedtime routine. My son has trouble with the transition from play time to bedtime. We have about 30 minutes, and I have a visual timer, some fidget toys, and his favorite books. In the past, visual timers have worked well for him."

In [ ]:
print_separator("📋 SCENARIO 2: Bedtime Routine Planning")

scenario_2 = """I need help creating a structured bedtime routine.
My son has trouble with the transition from play time to bedtime.
We have about 30 minutes, and I have a visual timer, some fidget toys, and his favorite books.
In the past, visual timers have worked well for him."""

print("Parent Question:")
print(f'"{scenario_2}"\n')
print("🤖 Coordinator orchestrating specialists...\n")

# Create session for scenario 2
await session_service.create_session(
    app_name="agents",
    user_id="demo_user",
    session_id="scenario_2",
)

# Run scenario with automatic retry
await run_scenario_with_retry(
    runner=runner,
    user_id="demo_user",
    session_id="scenario_2",
    message=genai.types.Content(parts=[genai.types.Part(text=scenario_2)]),
)

print("\n✅ Scenario 2 complete!")

### 🔍 What Just Happened?

1. **Coordinator identified** activity planning need
2. **Delegated** to Activity Planner agent
3. **Activity Planner**:
   - Used ActivityPlanner custom tool
   - Created structured plan with:
     - Complete materials list
     - Environmental setup instructions
     - Step-by-step timing
     - Observable success criteria
     - ADHD/ASD adaptations
4. **Memory Agent** (if called):
   - Recalled that visual timers worked well before
   - Integrated past successes into plan

**ADK Capabilities Demonstrated:**
- ✅ Custom function tools (ActivityPlanner)
- ✅ Structured data output (Pydantic models)
- ✅ Agent specialization
- ✅ Context integration (past experiences)

In [ ]:
# Add delay between scenarios
print("\n⏳ Waiting 2 seconds before next scenario...\n")
await asyncio.sleep(2)

---

## 📋 Demo Scenario 3: Learning from Past Experiences

**Demonstrates:**
- Cross-session learning and pattern recognition
- Memory Agent with PatternAnalyzer tool
- Personalization based on what works for THIS child
- Data-driven recommendations

**Parent Question:**
> "Can you analyze what's been working for bedtime? Here's our history:
> 
> Session 1: Used visual timer (10 min), worked well, he went to bed without resistance
> Session 2: Just verbal reminders, didn't work, he got upset and it took 45 minutes
> Session 3: Visual timer + sensory break before bed, worked great, he was calm
> 
> What patterns do you see?"

In [ ]:
print_separator("📋 SCENARIO 3: Learning from Past Experiences")

scenario_3 = """Can you analyze what's been working for bedtime? Here's our history:

Session 1: Used visual timer (10 min), worked well, he went to bed without resistance
Session 2: Just verbal reminders, didn't work, he got upset and it took 45 minutes
Session 3: Visual timer + sensory break before bed, worked great, he was calm

What patterns do you see?"""

print("Parent Question:")
print(f'"{scenario_3}"\n')
print("🤖 Coordinator orchestrating specialists...\n")

# Create session for scenario 3
await session_service.create_session(
    app_name="agents",
    user_id="demo_user",
    session_id="scenario_3",
)

# Run scenario with automatic retry
await run_scenario_with_retry(
    runner=runner,
    user_id="demo_user",
    session_id="scenario_3",
    message=genai.types.Content(parts=[genai.types.Part(text=scenario_3)]),
)

print("\n✅ Scenario 3 complete!")

### 🔍 What Just Happened?

1. **Coordinator identified** pattern learning request
2. **Delegated** to Memory Agent
3. **Memory Agent**:
   - Used PatternAnalyzer custom tool
   - Analyzed session outcomes (what worked vs. what didn't)
   - Identified patterns:
     - ✅ Visual timers: 2/2 success rate
     - ❌ Verbal reminders only: 0/1 success rate
     - ✅ Sensory breaks: 1/1 success rate
   - Generated data-driven recommendations
4. **Coordinator** might consult ADHD/ASD experts for why these patterns work

**ADK Capabilities Demonstrated:**
- ✅ Cross-session learning
- ✅ Pattern recognition
- ✅ Personalization
- ✅ Data-driven insights
- ✅ Custom tools (PatternAnalyzer)

---

## 🎉 Demo Complete!

### Summary of Demonstrated Capabilities

This capstone project successfully demonstrates:

#### ✅ Day 1: Multi-Agent Orchestration
- **Hierarchical system**: 1 Manager (Coordinator) + 5 Specialists
- **AgentTool pattern**: Specialists wrapped as tools for the coordinator
- **Intelligent routing**: Coordinator delegates to appropriate specialists

#### ✅ Day 2: Tools & Integration
- **GoogleSearchTool**: Evidence-based research from CHADD, CDC, Autism Speaks
- **Custom FunctionTools**:
  - BehaviorClassifier: Analyzes ADHD/ASD/age-typical factors
  - ActivityPlanner: Creates structured activity plans
  - PatternAnalyzer: Learns from session history
- **Type-safe interfaces**: All tools use Pydantic v2 models

#### ✅ Day 3: Memory & Context
- **Pattern learning**: Memory Agent analyzes what works for THIS child
- **Personalization**: Recommendations based on past successes
- **Cross-session data**: Can incorporate history from multiple sessions

#### ✅ Day 4: Session Management
- **InMemorySessionService**: Conversation state tracking
- **Session IDs**: Separate contexts for different scenarios
- **Context preservation**: Agents maintain conversation history

#### ✅ Day 5: Production-Ready
- **Error handling**: Two-layer retry strategy (agent + application level)
- **ADK best practices**: HttpRetryOptions with exponential backoff
- **Type safety**: Strict Pydantic models, no `Dict[str, Any]`
- **Rate limit handling**: Automatic retries with jitter

### Real-World Impact

This system helps parents:
1. **Understand behaviors** - Is it ADHD, autism, or typical development?
2. **Access evidence-based strategies** - Research-backed interventions
3. **Create structured plans** - Complete activity plans with materials
4. **Learn what works** - Personalized insights from their own experiences

### Technical Excellence

- **Type-safe architecture**: 0 `Dict[str, Any]` in domain code
- **Discriminated unions**: All results are Success OR Error, never mixed
- **Enum-driven constants**: No string typos possible
- **Field validation**: Automatic validation on construction and assignment
- **Production-ready**: Comprehensive error handling and retry logic

---

## 🔮 Future Enhancements

### Phase 1: Enhanced Memory
- Persistent storage with database backend
- Long-term pattern tracking across weeks/months
- Behavioral trend analysis

### Phase 2: Expanded Agents
- **Sensory Processing Specialist**: Deep dive into sensory needs
- **Social Skills Coach**: Peer interaction strategies
- **School Liaison**: IEP/504 plan guidance

### Phase 3: Interactive Tools
- Visual schedule generator
- Behavior tracking dashboard
- Progress reports for healthcare providers

### Phase 4: Community Features
- Anonymous pattern sharing (privacy-preserving)
- Evidence-based strategy library
- Parent community forum integration

---

## ⚠️ Important Disclaimer

**This is a parenting support tool, NOT medical advice.**

Always consult qualified healthcare professionals for:
- Medical diagnoses
- Treatment plans
- Medication decisions
- Serious behavioral concerns

This tool provides:
- ✅ Evidence-based strategy suggestions
- ✅ Activity planning support
- ✅ Pattern recognition from your data
- ✅ Educational resources

This tool does NOT provide:
- ❌ Medical diagnoses
- ❌ Prescriptions or treatment plans
- ❌ Emergency assistance
- ❌ Substitute for professional care

---

## 📚 Resources

### For Parents
- [CHADD](https://chadd.org/) - ADHD support and research
- [Autism Speaks](https://www.autismspeaks.org/) - ASD resources
- [CDC Developmental Milestones](https://www.cdc.gov/ncbddd/actearly/milestones/index.html)
- [Russell Barkley's Work](https://www.russellbarkley.org/) - ADHD expert

### For Developers
- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [Pydantic Documentation](https://docs.pydantic.dev/)
- [Project Repository](https://github.com/marcuskbra/neuro-diver-agents)

---

## 🙏 Acknowledgments

- **Google & Kaggle** for the ADK framework and Agents Intensive Course
- **CHADD, Autism Speaks, CDC** for evidence-based resources
- **Parents of neurodivergent children** for inspiring this meaningful use case
- **Neurodivergent community** for advocacy and awareness

---

## 📝 License

MIT License - This is an educational project for the Kaggle Agents Intensive Capstone.

---

**Built with ❤️ for neurodivergent children and their families**